[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_15/2_sql.ipynb)

# Day 15: SQL (medium)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (30 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Who reports to whom | medium | 5 |
| Q2 | Genres bought in the same order | medium | 6 |
| Q3 | Do buyers bring buyers? | medium | 7 |
| Q4 | Fans of both | medium | 7 |
| Q5 | Referral share of the user base, week by week (review) | medium | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Real data: the Chinook digital music store (copied into the in-memory db).
_src = sqlite3.connect(data_path("chinook.sqlite", "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"))
_src.backup(db)
_src.close()

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: MovieLens small (GroupLens, research and education use).
def movielens(short):
    path = os.path.join(DATA, f"movielens_{short}.csv")
    if not os.path.exists(path):
        import io, zipfile
        print("downloading MovieLens")
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip").read()))
        for s in ("ratings", "movies", "tags", "links"):
            with open(os.path.join(DATA, f"movielens_{s}.csv"), "wb") as f:
                f.write(z.read(f"ml-latest-small/{s}.csv"))
    return pd.read_csv(path)

_r = movielens("ratings")
_r["rated_at"] = pd.to_datetime(_r["timestamp"], unit="s").dt.strftime("%Y-%m-%d %H:%M:%S")
_r.rename(columns={"userId": "user_id", "movieId": "movie_id"})[["user_id", "movie_id", "rating", "rated_at"]] \
    .to_sql("ml_ratings", db, index=False)
movielens("movies").rename(columns={"movieId": "movie_id"}).to_sql("ml_movies", db, index=False)

print("tables:", ", ".join(r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

## Tables

**Real tables: Chinook** (a digital music store, 2009 to 2013; MIT license, lerocha/chinook-database):

| Table | One row is | Key columns |
|---|---|---|
| `Customer` | one customer (59) | `CustomerId`, `FirstName`, `LastName`, `Company` (often NULL), `City`, `State` (often NULL), `Country`, `Email`, `SupportRepId` |
| `Employee` | one employee (8) | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` (manager id, NULL for the boss), `HireDate`, `City` |
| `Invoice` | one purchase (412) | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingCity`, `BillingCountry`, `Total` |
| `InvoiceLine` | one track on an invoice (2,240) | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` |
| `Track` | one song or video (3,503) | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer` (can be NULL), `Milliseconds`, `Bytes`, `UnitPrice` |
| `Album` / `Artist` | one album (347) / artist (275) | `AlbumId`, `Title`, `ArtistId` / `ArtistId`, `Name` |
| `Genre` / `MediaType` | one genre (25) / format (5) | `GenreId`, `Name` / `MediaTypeId`, `Name` |
| `Playlist` / `PlaylistTrack` | one playlist (18) / one track in a playlist (8,715) | `PlaylistId`, `Name` / `PlaylistId`, `TrackId` |

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real tables: MovieLens small** (GroupLens; 610 users rated 9,742 movies, 1996 to 2018):

| Table | One row is | Columns |
|---|---|---|
| `ml_ratings` | one rating (100,836) | `user_id`, `movie_id`, `rating` (0.5 to 5), `rated_at` ('YYYY-MM-DD HH:MM:SS', UTC) |
| `ml_movies` | one movie (9,742) | `movie_id`, `title` (with the year), `genres` (several genres joined by a vertical bar, e.g. Comedy\|Romance) |

The queries run in SQLite. They are written in standard SQL that also works in PostgreSQL; differences (dates, integer division) are noted in the solutions.

---
### Q1. Who reports to whom

*medium, about 5 min*

Chinook `Employee` has a `ReportsTo` column with the id of each person's manager (NULL for the top
boss). Return all 8 employees with `EmployeeId`, `employee` (full name), `manager` (full name, NULL for the boss),
`skip_manager` (the manager's manager) and `direct_reports` (how many people report to this employee, 0 if nobody).

Example row: `3 | Jane Peacock | Nancy Edwards | Andrew Adams | 0`.

Follow-up: how would you list everybody under Andrew Adams at any depth?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: a table that points at itself (`ReportsTo` -> `EmployeeId`). Pattern: self join with aliases; LEFT JOIN so the boss (NULL manager) stays.

</details>

<details><summary><b>Hint 2</b></summary>

1. `Employee e LEFT JOIN Employee m ON m.EmployeeId = e.ReportsTo` (manager).
2. `LEFT JOIN Employee mm ON mm.EmployeeId = m.ReportsTo` (skip level).
3. `LEFT JOIN Employee r ON r.ReportsTo = e.EmployeeId` and COUNT(r.EmployeeId), grouped by employee.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT e.EmployeeId,
       e.FirstName || ' ' || e.LastName   AS employee,
       m.FirstName || ' ' || m.LastName   AS manager,
       mm.FirstName || ' ' || mm.LastName AS skip_manager,
       COUNT(r.EmployeeId)                AS direct_reports
FROM Employee e
LEFT JOIN Employee m  ON m.EmployeeId  = e.ReportsTo     -- my manager
LEFT JOIN Employee mm ON mm.EmployeeId = m.ReportsTo     -- my manager's manager
LEFT JOIN Employee r  ON r.ReportsTo   = e.EmployeeId    -- people who report to me
GROUP BY e.EmployeeId, e.FirstName, e.LastName, m.FirstName, m.LastName, mm.FirstName, mm.LastName
ORDER BY e.EmployeeId
```

Result:

| EmployeeId | employee | manager | skip_manager | direct_reports |
|---|---|---|---|---|
| 1 | Andrew Adams | NULL | NULL | 2 |
| 2 | Nancy Edwards | Andrew Adams | NULL | 3 |
| 3 | Jane Peacock | Nancy Edwards | Andrew Adams | 0 |
| 4 | Margaret Park | Nancy Edwards | Andrew Adams | 0 |
| 5 | Steve Johnson | Nancy Edwards | Andrew Adams | 0 |
| 6 | Michael Mitchell | Andrew Adams | NULL | 2 |
| 7 | Robert King | Michael Mitchell | Andrew Adams | 0 |
| 8 | Laura Callahan | Michael Mitchell | Andrew Adams | 0 |

**Why:** Each alias is a separate copy of the same table playing a different role (me, my manager, my skip manager, my reports); the join condition says which role. LEFT JOIN keeps Andrew Adams, whose ReportsTo is NULL, and also keeps people without reports. `COUNT(r.EmployeeId)` counts only matched rows, so it gives 0, while `COUNT(*)` would give 1. In SQLite `'a' || NULL` is NULL, so the boss gets a NULL manager name, as asked. Follow-up: any depth needs a recursive CTE (day 24).

**Complexity:** 8 rows; in general one index lookup per join on the id columns.

**Common mistakes:** Swapping the direction (`m.ReportsTo = e.EmployeeId` finds reports, not the manager). Inner joins that drop the boss. `COUNT(*)` for direct reports.

**Learn more:** [sql-self-join](https://mahsa-agz.github.io/ds-handbook/#sql-self-join), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls), [sql-recursive-cte](https://mahsa-agz.github.io/ds-handbook/#sql-recursive-cte)

</details>

---
### Q2. Genres bought in the same order

*medium, about 6 min*

Chinook. Two genres are **bought together** when both appear on the same invoice. Return the
**8 genre pairs** that appear together on the most invoices: `genre_a`, `genre_b`, `invoices`, most first (ties:
alphabetical). List each pair once, with `genre_a` alphabetically before `genre_b`.

Example: an invoice with 3 Rock tracks, 1 Jazz track and 2 Latin tracks adds 1 to (Jazz, Latin), (Jazz, Rock) and
(Latin, Rock). It does not add 3 to anything.

Follow-up: Rock appears in most top pairs simply because it is the biggest genre. How would you measure
"surprisingly often together"?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "pairs that occur together in the same basket". Pattern: self join of a (basket, item) table on the basket id, with `a.item < b.item` to avoid duplicates and self pairs.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: `SELECT DISTINCT InvoiceId, genre` (one row per genre per invoice).
2. Join the CTE to itself `ON a.InvoiceId = b.InvoiceId AND a.genre < b.genre`.
3. Group by the pair, COUNT(*), order, LIMIT 8.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH inv_genre AS (
    SELECT DISTINCT il.InvoiceId, g.Name AS genre
    FROM InvoiceLine il
    JOIN Track t ON t.TrackId = il.TrackId
    JOIN Genre g ON g.GenreId = t.GenreId
)
SELECT a.genre AS genre_a, b.genre AS genre_b, COUNT(*) AS invoices
FROM inv_genre a
JOIN inv_genre b
  ON a.InvoiceId = b.InvoiceId
 AND a.genre < b.genre
GROUP BY a.genre, b.genre
ORDER BY invoices DESC, genre_a, genre_b
LIMIT 8
```

Result:

| genre_a | genre_b | invoices |
|---|---|---|
| Alternative & Punk | Rock | 54 |
| Latin | Rock | 51 |
| Metal | Rock | 46 |
| Latin | Metal | 25 |
| Jazz | Rock | 24 |
| Alternative & Punk | Latin | 20 |
| Jazz | Latin | 17 |
| Jazz | Metal | 15 |

**Why:** The DISTINCT step turns lines into one row per (invoice, genre), so 3 Rock tracks count once. The condition `a.genre < b.genre` removes (Rock, Rock) and keeps only one of (Jazz, Rock) / (Rock, Jazz); `<>` would list every pair twice. Follow-up: use lift = `P(A and B) / (P(A) * P(B))` with P = share of invoices. Lift above 1 means more often together than chance. Big genres get high counts but not necessarily high lift.

**Complexity:** The self join is quadratic in the number of genres per invoice (small here). On big baskets, restrict to frequent items first.

**Common mistakes:** Joining raw invoice lines (inflated counts). `a.genre <> b.genre` (double counting). Forgetting the invoice condition, which pairs every row with every row (a cross join).

**Learn more:** [sql-self-join](https://mahsa-agz.github.io/ds-handbook/#sql-self-join), [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning)

</details>

---
### Q3. Do buyers bring buyers?

*medium, about 7 min*

Shopping app (made up). `app_users.referred_by` holds the user_id of the person who invited this user.
Growth asks: **are friends invited by buyers more likely to buy?** A buyer is a user with at least one completed order.

Return one row per `referrer_type` ('buyer' or 'non_buyer'): `referred_users`, `referred_buyers`, `pct_bought`
(1 decimal) and `pct_same_platform` (share of invited friends on the same platform as their referrer, 1 decimal).

Then say in one sentence whether you would tell Growth "buyers bring buyers".

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: a column that points to another row of the same table (`referred_by`). Pattern: self join friend -> referrer, plus LEFT JOINs to a buyer list for both sides.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE buyers: DISTINCT user_id with a completed order.
2. `app_users f JOIN app_users r ON r.user_id = f.referred_by`.
3. LEFT JOIN buyers twice (once for r, once for f); CASE to label and flag; group by referrer type.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH buyers AS (
    SELECT DISTINCT user_id FROM purchases WHERE status = 'completed'
), pairs AS (
    SELECT f.user_id AS friend_id,
           r.user_id AS referrer_id,
           CASE WHEN rb.user_id IS NOT NULL THEN 'buyer' ELSE 'non_buyer' END AS referrer_type,
           CASE WHEN fb.user_id IS NOT NULL THEN 1 ELSE 0 END AS friend_bought,
           CASE WHEN f.platform = r.platform THEN 1 ELSE 0 END AS same_platform
    FROM app_users f
    JOIN app_users r     ON r.user_id = f.referred_by
    LEFT JOIN buyers rb  ON rb.user_id = r.user_id
    LEFT JOIN buyers fb  ON fb.user_id = f.user_id
)
SELECT referrer_type,
       COUNT(*)                            AS referred_users,
       SUM(friend_bought)                  AS referred_buyers,
       ROUND(100.0 * AVG(friend_bought), 1) AS pct_bought,
       ROUND(100.0 * AVG(same_platform), 1) AS pct_same_platform
FROM pairs
GROUP BY referrer_type
```

Result:

| referrer_type | referred_users | referred_buyers | pct_bought | pct_same_platform |
|---|---|---|---|---|
| buyer | 65 | 20 | 30.8 | 44.6 |
| non_buyer | 162 | 40 | 24.7 | 39.5 |

**Why:** The self join puts each invited friend next to their referrer, so attributes of both can be compared on one row. 30.8% of friends of buyers bought versus 24.7% of friends of non-buyers, but with only 65 friends in the buyer group the gap is within noise (a two-proportion test would not be significant), and it is correlation, not causation. Tell Growth: "a small, uncertain difference; not enough to target referral rewards at buyers yet". AVG of a 0/1 flag is a rate, a handy trick.

**Complexity:** Index lookups on user_id for each join; linear in the number of referred users.

**Common mistakes:** Joining in the wrong direction (`r.referred_by = f.user_id` finds who referred the referrer). Inner joins to buyers (drops non-buyers). Over-reading a small difference.

**Learn more:** [sql-self-join](https://mahsa-agz.github.io/ds-handbook/#sql-self-join), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [stats-hypothesis-tests](https://mahsa-agz.github.io/ds-handbook/#stats-hypothesis-tests)

</details>

---
### Q4. Fans of both

*medium, about 7 min*

MovieLens (real). Take the **15 most rated movies** (ties by movie_id). A user **likes** a movie if
they rated it 4 or more. Which **5 pairs** of these movies have the most users who like **both**?

Return `movie_a`, `movie_b` (titles, each pair once), `fans_of_both`, most first.

Example: if user 1 likes Pulp Fiction, Forrest Gump and Fight Club, they add 1 to three pairs.

Follow-up: why restrict to the top 15 first? What would happen on all 9,742 movies?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "pairs of items that share users". Pattern: self join of (user, movie) on user_id with `a.movie_id < b.movie_id`, after filtering both the movies and the likes.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE top: 15 movie_ids by COUNT(*) DESC.
2. CTE liked: ratings >= 4 on those movies.
3. Self join liked on user_id with a.movie_id < b.movie_id, group by the pair, join titles, LIMIT 5.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH top AS (
    SELECT movie_id
    FROM ml_ratings
    GROUP BY movie_id
    ORDER BY COUNT(*) DESC, movie_id
    LIMIT 15
), liked AS (
    SELECT r.user_id, r.movie_id
    FROM ml_ratings r
    JOIN top t ON t.movie_id = r.movie_id
    WHERE r.rating >= 4
)
SELECT ma.title AS movie_a, mb.title AS movie_b, COUNT(*) AS fans_of_both
FROM liked a
JOIN liked b     ON a.user_id = b.user_id AND a.movie_id < b.movie_id
JOIN ml_movies ma ON ma.movie_id = a.movie_id
JOIN ml_movies mb ON mb.movie_id = b.movie_id
GROUP BY a.movie_id, b.movie_id, ma.title, mb.title
ORDER BY fans_of_both DESC, movie_a
LIMIT 5
```

Result:

| movie_a | movie_b | fans_of_both |
|---|---|---|
| Shawshank Redemption, The (1994) | Forrest Gump (1994) | 168 |
| Pulp Fiction (1994) | Shawshank Redemption, The (1994) | 158 |
| Pulp Fiction (1994) | Silence of the Lambs, The (1991) | 152 |
| Star Wars: Episode IV - A New Hope (1977) | Star Wars: Episode V - The Empire Strikes Back (1980) | 144 |
| Shawshank Redemption, The (1994) | Silence of the Lambs, The (1991) | 140 |

**Why:** This is item-to-item co-occurrence, the simplest "people who liked X also liked Y" recommender. Filtering before the self join keeps it small: a user who likes k of the 15 movies creates k(k-1)/2 pairs. On the full table, heavy users with thousands of ratings would create millions of pairs, so in production you limit items, sample users, or compute it in a batch job (or with embeddings). Raw co-counts favour popular movies; cosine similarity or lift normalises for popularity.

**Complexity:** Quadratic in likes per user for the self join; 2,860 liked rows here.

**Common mistakes:** Self joining before filtering (slow). `a.movie_id <> b.movie_id` (each pair twice). Grouping by title only (two movies can share a title).

**Learn more:** [sql-self-join](https://mahsa-agz.github.io/ds-handbook/#sql-self-join), [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [ai-recommenders](https://mahsa-agz.github.io/ds-handbook/#ai-recommenders)

</details>

---
### Q5. Referral share of the user base, week by week (review)

*medium, about 5 min*

Shopping app. Weeks start on Monday (`date(d, 'weekday 0', '-6 days')`). For each signup week return
`week_start`, `signups`, `referral_signups`, `cum_users`, `cum_referral` (both cumulative since launch) and
`cum_referral_pct` = cum_referral / cum_users in percent (1 decimal).

Example: weeks with (signups, referral) = (100, 20) then (300, 30) give cumulative shares 20.0 and 12.5.
The average of the weekly rates (20% and 10%) would wrongly say 15.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: two running totals, then divide them. Never average weekly ratios to get a cumulative ratio.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE weekly: COUNT(*) and SUM(CASE WHEN channel = 'referral' ...) per week.
2. `SUM(x) OVER (ORDER BY week_start ROWS UNBOUNDED PRECEDING)` for both; divide with 100.0.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH weekly AS (
    SELECT date(signup_date, 'weekday 0', '-6 days') AS week_start,
           COUNT(*) AS signups,
           SUM(CASE WHEN channel = 'referral' THEN 1 ELSE 0 END) AS referral_signups
    FROM app_users
    GROUP BY date(signup_date, 'weekday 0', '-6 days')
)
SELECT week_start, signups, referral_signups,
       SUM(signups)          OVER w AS cum_users,
       SUM(referral_signups) OVER w AS cum_referral,
       ROUND(100.0 * SUM(referral_signups) OVER w / SUM(signups) OVER w, 1) AS cum_referral_pct
FROM weekly
WINDOW w AS (ORDER BY week_start ROWS UNBOUNDED PRECEDING)
ORDER BY week_start
```

Result:

| week_start | signups | referral_signups | cum_users | cum_referral | cum_referral_pct |
|---|---|---|---|---|---|
| 2024-01-01 | 174 | 27 | 174 | 27 | 15.5 |
| 2024-01-08 | 179 | 32 | 353 | 59 | 16.7 |
| 2024-01-15 | 170 | 33 | 523 | 92 | 17.6 |
| 2024-01-22 | 181 | 26 | 704 | 118 | 16.8 |
| 2024-01-29 | 175 | 22 | 879 | 140 | 15.9 |
| 2024-02-05 | 168 | 16 | 1047 | 156 | 14.9 |
| 2024-02-12 | 200 | 33 | 1247 | 189 | 15.2 |
| 2024-02-19 | 156 | 25 | 1403 | 214 | 15.3 |
| 2024-02-26 | 97 | 13 | 1500 | 227 | 15.1 |

**Why:** A ratio of running totals weights every user equally; an average of weekly ratios weights every week equally, which is wrong when week sizes differ (the last week is half size). The share is stable around 15% to 17%, and the final cumulative value (227 of 1,500) equals the overall referral share.

**Complexity:** One group by and windows over 9 rows.

**Common mistakes:** `AVG(weekly_pct) OVER (...)` (average of ratios). Integer division. Forgetting the frame and relying on the default (fine here because weeks are unique, risky otherwise).

**Learn more:** [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_15/3_stats.ipynb)